## algorithm design and anlysis-2026 spring  homework 2
**Deadline**：2026.5.20

**name**:


note：
---
1. 本题目为在线OJ作业，OJ平台题目链接：https://www.nowcoder.com/acm/contest/129481，访问密码见课件；
3. 在OJ平台运行通过后，将源码复制到本文件对应题目的下方代码框中；
4. 如若作答有雷同，全部取消成绩；


## A 排序

In [ ]:
import sys
from functools import lru_cache

MAX_OPS = 32768


class TooManyOperations(Exception):
    pass


def build_perm_ops(vals):

    m = len(vals)
    
    seen = [False] * m
    for x in vals:
        if x < 0 or x >= m or seen[x]:
            return None
        seen[x] = True

    if m == 1:
        return []

    half = m >> 1
    left = [vals[i << 1] >> 1 for i in range(half)]
    right = [vals[(i << 1) | 1] >> 1 for i in range(half)]

    left_ops = build_perm_ops(left)
    if left_ops is None:
        return None
    right_ops = build_perm_ops(right)
    if right_ops is None:
        return None

    ops = []

    if vals[0] & 1:
        ops.append(1 if m == 2 else -1)

    cur_xor_l = 0
    for x in left_ops:
        if x > 0:
            ops.append(-1)
            ops.append(1)
        else:
            ops.append(x << 1)
            cur_xor_l ^= (-x) << 1

    if cur_xor_l:
        ops.append(-cur_xor_l)

    cur_xor_r = 0
    for x in right_ops:
        if x > 0:
            ops.append(1)
            ops.append(-1)
        else:
            ops.append(x << 1)
            cur_xor_r ^= (-x) << 1

    if (cur_xor_r & half) != (cur_xor_l & half):
        return None

    if cur_xor_l >= half:
        cur_xor_l -= half
    if cur_xor_r >= half:
        cur_xor_r -= half
    if cur_xor_l != cur_xor_r:
        return None

    merged = []
    for x in ops:
        if merged and x < 0 and merged[-1] < 0:
            merged_xor = (-merged[-1]) ^ (-x)
            if merged_xor:
                merged[-1] = -merged_xor
            else:
                merged.pop()
        else:
            merged.append(x)

    return merged


def solve():
    data = list(map(int, sys.stdin.buffer.read().split()))
    if not data:
        return

    n, A, B = data[0], data[1], data[2]
    p = data[3:3 + n]
    mask = n - 1
    answer = []

    def emit(x):
        answer.append(x)
        if len(answer) > MAX_OPS:
            raise TooManyOperations

    def emit_add(v):
        v &= mask
        if v:
            emit(v)

    def emit_xor(v):
        if v:
            emit(-v)

    def emit_swap_magic():
        emit(0)

    def rebuild_pos():
        pos = [0] * n
        for i, v in enumerate(p):
            pos[v] = i
        return pos

    def do_add_magic(v):
        nonlocal p, pos
        v &= mask
        if not v:
            return
        emit(v)
        p = [(x + v) & mask for x in p]
        pos = rebuild_pos()

    def do_xor_magic(v):
        nonlocal p, pos
        if not v:
            return
        emit(-v)
        p = [x ^ v for x in p]
        pos = rebuild_pos()

    lowbit_len = (A - B + n) & mask
    lowbit_len &= -lowbit_len
    if lowbit_len == 0:
        lowbit_len = n

    @lru_cache(maxsize=None)
    def get_pair_pos(x, y):
        delta = (y - x + n - lowbit_len + n) & mask
        px = 0
        py = 0
        step = n >> 1
        while step >= (lowbit_len << 1):
            if delta >= step:
                delta -= step
                py += step >> 1
            else:
                px += step >> 1
            step >>= 1

        low = x & (lowbit_len - 1)
        px += (n >> 1) + low
        py += low
        return px, py

    pA, _ = get_pair_pos(A, B)
    pos = rebuild_pos()

    def swap_values(x, y):
        nonlocal p, pos
        ix = pos[x]
        iy = pos[y]
        p[ix], p[iy] = p[iy], p[ix]
        pos[x], pos[y] = iy, ix

    sys.setrecursionlimit(10000)

    def emit_direct_swap_and_update(x, y):
        
        pX, _ = get_pair_pos(x, y)

        emit_add((pX - x) & mask)
        emit_xor(pX ^ pA)
        emit_add((A - pA) & mask)

        emit_swap_magic()

        emit_add((pA - A) & mask)
        emit_xor(pX ^ pA)
        emit_add((x - pX) & mask)

        swap_values(x, y)

    def apply_swap(x, y):
        if ((x // lowbit_len) & 1) == ((y // lowbit_len) & 1):
            if ((x // lowbit_len) & 1) == 0:
                mid = (x & (lowbit_len - 1)) + lowbit_len
            else:
                mid = x & (lowbit_len - 1)
            apply_swap(x, mid)
            apply_swap(y, mid)
            apply_swap(x, mid)
            return

        emit_direct_swap_and_update(x, y)

    try:
        if lowbit_len > 1:
            base_vals = [p[i] & (lowbit_len - 1) for i in range(lowbit_len)]
            base_ops = build_perm_ops(base_vals)
            if base_ops is None:
                print(-1)
                return

            for x in base_ops:
                if x > 0:
                    do_add_magic(x)
                else:
                    do_xor_magic(-x)

        for rem in range(lowbit_len):
            group = p[rem:n:lowbit_len]
            group.sort()
            if group != list(range(rem, n, lowbit_len)):
                print(-1)
                return

            j = rem
            while j < n:
                if p[j] != j:
                    apply_swap(j, p[j])
                j += lowbit_len

        if p != list(range(n)):
            print(-1)
            return

    except TooManyOperations:
        print(-1)
        return

    out = [str(len(answer))]
    for x in answer:
        if x == 0:
            out.append("0")
        elif x < 0:
            out.append(f"1 {-x}")
        else:
            out.append(f"2 {x}")

    sys.stdout.write("\n".join(out))


if __name__ == "__main__":
    solve()


## B 长跑

In [ ]:
import sys


def can_finish(n: int, length: int, max_energy: int, coins: int, stations: list[tuple[int, int]]) -> bool:
	if length <= max_energy:
		return True

	if n == 0:
		return False

	best_cost_at_pos: dict[int, int] = {}
	for pos, cost in stations:
		if 0 <= pos <= length:
			if pos not in best_cost_at_pos or cost < best_cost_at_pos[pos]:
				best_cost_at_pos[pos] = cost

	if not best_cost_at_pos:
		return False

	pos_cost = sorted(best_cost_at_pos.items())
	m = len(pos_cost)
	inf = 10**18
	dp = [inf] * m

	for i, (pos, cost) in enumerate(pos_cost):
		if pos <= max_energy:
			dp[i] = cost

	for i in range(m):
		if dp[i] > coins:
			continue
		cur_pos, _ = pos_cost[i]
		if length - cur_pos <= max_energy:
			return True

		j = i + 1
		while j < m and pos_cost[j][0] - cur_pos <= max_energy:
			next_cost = dp[i] + pos_cost[j][1]
			if next_cost < dp[j]:
				dp[j] = next_cost
			j += 1

	return False


def solve() -> None:
	data = list(map(int, sys.stdin.buffer.read().split()))
	idx = 0
	out = []

	while idx + 3 < len(data):
		n = data[idx]
		length = data[idx + 1]
		max_energy = data[idx + 2]
		coins = data[idx + 3]
		idx += 4

		if idx + 2 * n > len(data):
			break

		stations = []
		for _ in range(n):
			pos = data[idx]
			cost = data[idx + 1]
			idx += 2
			stations.append((pos, cost))

		out.append("Yes" if can_finish(n, length, max_energy, coins, stations) else "No")

	sys.stdout.write("\n".join(out))


if __name__ == "__main__":
	solve()


## C 最长回文

In [ ]:
#include <bits/stdc++.h>
using namespace std;

using ull = unsigned long long;

static const ull BASE1 = 911382323ULL;
static const ull BASE2 = 972663749ULL;

static vector<int> manacher(const string& t) {
    int n = (int)t.size();
    vector<int> p(n, 0);
    int center = 0, right = 0;
    for (int i = 1; i < n; ++i) {
        if (i < right) {
            int mirror = center * 2 - i;
            p[i] = min(p[mirror], right - i);
        } else {
            p[i] = 1;
        }
        while (i - p[i] >= 0 && i + p[i] < n && t[i - p[i]] == t[i + p[i]]) {
            ++p[i];
        }
        if (i + p[i] > right) {
            center = i;
            right = i + p[i];
        }
    }
    return p;
}

struct Hash64 {
    vector<ull> pref;
    vector<ull> pw;

    Hash64() = default;
    Hash64(const string& s, ull base) {
        int n = (int)s.size();
        pref.assign(n + 1, 0);
        pw.assign(n + 1, 1);
        for (int i = 0; i < n; ++i) {
            pw[i + 1] = pw[i] * base;
            pref[i + 1] = pref[i] * base + (unsigned char)s[i];
        }
    }

    ull get(int l, int r) const {
        if (l > r) return 0;
        return pref[r + 1] - pref[l] * pw[r - l + 1];
    }
};

static bool same_segment(
    const Hash64& ha1, const Hash64& ha2,
    const Hash64& hr1, const Hash64& hr2,
    int m, int l1, int r1, int l2, int r2
) {
    if (l1 > r1) return true;
    int rl = m - 1 - r2;
    int rr = m - 1 - l2;
    return ha1.get(l1, r1) == hr1.get(rl, rr) &&
           ha2.get(l1, r1) == hr2.get(rl, rr);
}

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    int n;
    string a, b;
    cin >> n >> a >> b;

    string ta = "$#";
    ta.reserve(2 * n + 2);
    for (char c : a) {
        ta.push_back(c);
        ta.push_back('#');
    }

    string tb = "$#";
    tb.reserve(2 * n + 2);
    for (char c : b) {
        tb.push_back(c);
        tb.push_back('#');
    }

    int m = (int)ta.size();
    vector<int> pa = manacher(ta);
    vector<int> pb = manacher(tb);

    Hash64 ha1(ta, BASE1), ha2(ta, BASE2);
    string rb = tb;
    reverse(rb.begin(), rb.end());
    Hash64 hr1(rb, BASE1), hr2(rb, BASE2);

    int ans = max(*max_element(pa.begin(), pa.end()), *max_element(pb.begin(), pb.end())) - 1;

    for (int i = 2; i < m; ++i) {
        int base = max(pa[i], pb[i - 2]);
        int sa = i - base + 1;
        int sb = (i - 2) + base - 1;
        int lo = 0, hi = min(sa, m - 1 - sb), add = 0;
        while (lo <= hi) {
            int mid = (lo + hi) >> 1;
            if (same_segment(ha1, ha2, hr1, hr2, m, sa - mid, sa - 1, sb + 1, sb + mid)) {
                add = mid;
                lo = mid + 1;
            } else {
                hi = mid - 1;
            }
        }
        ans = max(ans, base + add - 1);
    }

    cout << ans << '\n';
    return 0;
}


## D 优惠券

In [ ]:
import sys


class Fenwick:
    def __init__(self, n: int):
        self.n = n
        self.bit = [0] * (n + 1)

    def add(self, i: int, delta: int) -> None:
        while i <= self.n:
            self.bit[i] += delta
            i += i & -i

    def sum(self, i: int) -> int:
        s = 0
        while i > 0:
            s += self.bit[i]
            i -= i & -i
        return s

    def kth(self, k: int) -> int:
        idx = 0
        bitmask = 1 << (self.n.bit_length() - 1)
        while bitmask:
            nxt = idx + bitmask
            if nxt <= self.n and self.bit[nxt] < k:
                idx = nxt
                k -= self.bit[nxt]
            bitmask >>= 1
        return idx + 1

    def consume_ge(self, pos: int) -> bool:
        before = self.sum(pos - 1)
        total = self.sum(self.n)
        if before == total:
            return False
        idx = self.kth(before + 1)
        self.add(idx, -1)
        return True


def solve() -> None:
    lines = sys.stdin.buffer.read().splitlines()
    idx = 0
    ans = []

    while idx < len(lines):
        line = lines[idx].strip()
        idx += 1
        if not line:
            continue

        m = int(line)
        records = []
        for _ in range(m):
            records.append(lines[idx].strip())
            idx += 1

        bit = Fenwick(m)
        hold = {}   
        lastI = {}
        lastO = {}

        bad = -1

        for i, rec in enumerate(records, 1):
            if rec == b"?" or rec == "？".encode():
                bit.add(i, 1)
                continue

            op, x = rec.split()
            x = int(x)

            if op == b"I":
                if hold.get(x, 0) == 0:
                    hold[x] = 1
                else:
                    if not bit.consume_ge(lastI.get(x, 0) + 1):
                        bad = i
                        break
                    
                    hold[x] = 1
                lastI[x] = i

            else:  
                if hold.get(x, 0) == 1:
                    hold[x] = 0
                else:
                    
                    if not bit.consume_ge(lastO.get(x, 0) + 1):
                        bad = i
                        break
                    hold[x] = 0
                lastO[x] = i

        ans.append(str(bad))

    sys.stdout.write("\n".join(ans))


if __name__ == "__main__":
    solve()

## E 任意点

In [ ]:
import sys


def solve():
    input = sys.stdin.readline
    n = int(input().strip())
    points = [tuple(map(int, input().split())) for _ in range(n)]

    visited = [False] * n

    def dfs(u):
        visited[u] = True
        x1, y1 = points[u]
        for v in range(n):
            if not visited[v]:
                x2, y2 = points[v]
                if x1 == x2 or y1 == y2:
                    dfs(v)

    components = 0
    for i in range(n):
        if not visited[i]:
            components += 1
            dfs(i)

    print(components - 1)


if __name__ == "__main__":
    solve()

## F 通配符匹配

In [ ]:
#include <bits/stdc++.h>
using namespace std;

struct Block {
    int len;  
    vector<pair<int, string>> segs;  
    int anchorOff;   
    string anchor;  
};

static Block buildBlock(const string& b) {
    Block res;
    res.len = (int)b.size();
    res.anchorOff = 0;
    res.anchor = "";

    int m = (int)b.size();
    for (int i = 0; i < m; ) {
        if (b[i] == '?') {
            ++i;
            continue;
        }
        int j = i;
        while (j < m && b[j] != '?') ++j;
        string lit = b.substr(i, j - i);
        res.segs.push_back({i, lit});
        if ((int)lit.size() > (int)res.anchor.size()) {
            res.anchor = lit;
            res.anchorOff = i;
        }
        i = j;
    }
    return res;
}

static bool matchAt(const string& s, int start, const Block& b) {
    if (start < 0 || start + b.len > (int)s.size()) return false;
    for (auto& seg : b.segs) {
        int off = seg.first;
        const string& lit = seg.second;
        if (s.compare(start + off, lit.size(), lit) != 0) return false;
    }
    return true;
}

static int findBlock(const string& s, int lo, int hi, const Block& b) {
   
    if (lo > hi) return -1;
    if (lo + b.len > (int)s.size()) return -1;

    
    if (b.segs.empty()) {
        return lo;
    }

    size_t searchPos = (size_t)(lo + b.anchorOff);

    while (true) {
        size_t occ = s.find(b.anchor, searchPos);
        if (occ == string::npos) return -1;

        int start = (int)occ - b.anchorOff;
        if (start > hi) return -1;

        if (start >= lo && matchAt(s, start, b)) {
            return start;
        }
        searchPos = occ + 1;
    }
}

static bool solveOne(const string& patternRaw, const string& s) {
    
    string pattern;
    pattern.reserve(patternRaw.size());
    for (char c : patternRaw) {
        if (c == '*' && !pattern.empty() && pattern.back() == '*') continue;
        pattern.push_back(c);
    }

    int stars = 0;
    for (char c : pattern) {
        if (c == '*') ++stars;
    }

    int minLen = (int)pattern.size() - stars;
    if ((int)s.size() < minLen) return false;

    
    if (stars == 0) {
        if (pattern.size() != s.size()) return false;
        for (int i = 0; i < (int)pattern.size(); ++i) {
            if (pattern[i] != '?' && pattern[i] != s[i]) return false;
        }
        return true;
    }

    
    vector<string> rawBlocks;
    {
        string cur;
        for (char c : pattern) {
            if (c == '*') {
                rawBlocks.push_back(cur);
                cur.clear();
            } else {
                cur.push_back(c);
            }
        }
        rawBlocks.push_back(cur);
    }

    vector<Block> blocks;
    blocks.reserve(rawBlocks.size());
    for (auto& b : rawBlocks) blocks.push_back(buildBlock(b));

    bool leadingStar = !pattern.empty() && pattern.front() == '*';
    bool trailingStar = !pattern.empty() && pattern.back() == '*';

    int left = 0;
    int right = (int)s.size();
    int L = 0;
    int R = (int)blocks.size() - 1;

    
    if (!leadingStar) {
        if (!matchAt(s, 0, blocks[0])) return false;
        left = blocks[0].len;
        ++L;
    }

    
    if (!trailingStar) {
        int start = (int)s.size() - blocks.back().len;
        if (start < left) return false;
        if (!matchAt(s, start, blocks.back())) return false;
        right = start;
        --R;
    }

    
    for (int i = L; i <= R; ++i) {
        if (blocks[i].len == 0) continue;  
        int pos = findBlock(s, left, right - blocks[i].len, blocks[i]);
        if (pos == -1) return false;
        left = pos + blocks[i].len;
    }

    return true;
}

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    string pattern;
    cin >> pattern;

    int n;
    cin >> n;

    while (n--) {
        string name;
        cin >> name;
        cout << (solveOne(pattern, name) ? "YES" : "NO") << '\n';
    }
    return 0;
}

## G 汉诺塔

In [ ]:
#include <bits/stdc++.h>
using namespace std;

long long qpow(long long a, int b) {
    long long res = 1;
    while (b--) res *= a;
    return res;
}

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    int n;
    cin >> n;

    vector<string> ops(6);
    for (int i = 0; i < 6; ++i) cin >> ops[i];

    unordered_map<string, int> rk;
    for (int i = 0; i < 6; ++i) rk[ops[i]] = i;

    unordered_map<char, char> nxt;

    nxt['A'] = (rk["AB"] < rk["AC"] ? 'B' : 'C');
    nxt['B'] = (rk["BA"] < rk["BC"] ? 'A' : 'C');
    nxt['C'] = (rk["CA"] < rk["CB"] ? 'A' : 'B');

    char a1 = nxt['A'];
    char a2 = nxt[a1];
    char a3 = nxt[a2];

    long long ans;

    if (a3 == 'A') {
        ans = qpow(2, n) - 1;
    } else if (a2 == 'A') {
        ans = 2 * qpow(3, n - 1) - 1;
    } else {
        ans = qpow(3, n - 1);
    }

    cout << ans << '\n';
    return 0;
}

## H 马步距离

In [ ]:
#include <bits/stdc++.h>
using namespace std;

long long knight_distance(long long x1, long long y1, long long x2, long long y2) {
    long long dx = llabs(x1 - x2);
    long long dy = llabs(y1 - y2);
    if (dx < dy) swap(dx, dy);

    
    if (dx == 1 && dy == 0) 
        return 3;
    if (dx == 2 && dy == 2) 
        return 4;

    long long d = max((dx + 1) / 2, (dx + dy + 2) / 3);

   
    if ((d & 1) != ((dx + dy) & 1)) 
        d++;

    return d;
}

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    long long xp, yp, xs, ys;
    cin >> xp >> yp >> xs >> ys;

    cout << knight_distance(xp, yp, xs, ys) << '\n';
    return 0;
}

## I 直方图最大矩形

In [ ]:
class Solution:
    def largestRectangleArea(self , heights: List[int]) -> int:
        
        stack = []
        ans = 0
        n = len(heights)

        for i in range(n + 1):
            cur = 0 if i == n else heights[i]

            while stack and heights[stack[-1]] >= cur:
                h = heights[stack.pop()]
                left = -1 if not stack else stack[-1]
                width = i - left - 1
                ans = max(ans, h * width)

            stack.append(i)

        return ans

## J 消防局的设立

In [ ]:
#include <bits/stdc++.h>
using namespace std;

class Solution {
public:
    int solve() {
        ios::sync_with_stdio(false);
        cin.tie(nullptr);

        int n;
        cin >> n;

        vector<int> fa(n + 1, 0), dep(n + 1, 0);
        for (int i = 2; i <= n; ++i) {
            cin >> fa[i];
            dep[i] = dep[fa[i]] + 1;
        }

        vector<int> order;
        order.reserve(n);
        for (int i = 1; i <= n; ++i) order.push_back(i);

        sort(order.begin(), order.end(), [&](int a, int b) {
            return dep[a] > dep[b];
        });

        vector<int> childSel(n + 1, 0), grandSel(n + 1, 0);
        vector<char> sel(n + 1, 0);

        auto covered = [&](int u) -> bool {
            int p = fa[u];
            int g = p ? fa[p] : 0;

            if (sel[u]) 
                return true;              
            if (p && sel[p]) 
                return true;         
            if (g && sel[g]) 
                return true;         
            if (childSel[u] > 0) 
                return true;     
            if (grandSel[u] > 0) 
                return true;     
            if (p && childSel[p] > 0) 
                return true; 

            return false;
        };

        int ans = 0;

        for (int u : order) {
            if (covered(u)) 
                continue;

            int v = u;
            if (fa[v]) v = fa[v];
            if (fa[v]) v = fa[v];   

            if (!sel[v]) {
                sel[v] = 1;
                ++ans;

                int p = fa[v];
                int g = p ? fa[p] : 0;

                if (p) childSel[p]++;
                if (g) grandSel[g]++;
            }
        }

        cout << ans << '\n';
        return 0;
    }
};

int main() {
    Solution().solve();
    return 0;
}